## Deep_clip inspired approach for evaluation (BAnG)

### Dataset used aptaDB   https://lmmd.ecust.edu.cn/aptadb/

In [ ]:
import pandas as pd
import random
import requests
import time
from transformers import AutoTokenizer, AutoModel, EsmTokenizer, EsmModel
import torch
import torch.nn as nn
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader




In [1]:
!wget https://lmmd.ecust.edu.cn/aptadb/download/aptamer.csv
!wget https://lmmd.ecust.edu.cn/aptadb/download/interaction.csv
!wget https://lmmd.ecust.edu.cn/aptadb/download/protein.csv

--2026-05-31 14:11:42--  https://lmmd.ecust.edu.cn/aptadb/download/aptamer.csv
Resolving lmmd.ecust.edu.cn (lmmd.ecust.edu.cn)... 59.78.109.156
Connecting to lmmd.ecust.edu.cn (lmmd.ecust.edu.cn)|59.78.109.156|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 352454 (344K) [text/csv]
Saving to: ‘aptamer.csv’

aptamer.csv         100%[===================>] 344.19K   444KB/s    in 0.8s    

2026-05-31 14:11:44 (444 KB/s) - ‘aptamer.csv’ saved [352454/352454]

--2026-05-31 14:11:44--  https://lmmd.ecust.edu.cn/aptadb/download/interaction.csv
Resolving lmmd.ecust.edu.cn (lmmd.ecust.edu.cn)... 59.78.109.156
Connecting to lmmd.ecust.edu.cn (lmmd.ecust.edu.cn)|59.78.109.156|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 218199 (213K) [text/csv]
Saving to: ‘interaction.csv’

interaction.csv     100%[===================>] 213.08K   265KB/s    in 0.8s    

2026-05-31 14:11:47 (265 KB/s) - ‘interaction.csv’ saved [218199/218199]

--2026-05-31 14:

In [ ]:

aptamer_df = pd.read_csv("/kaggle/working/aptamer.csv",encoding="latin1")
interaction_df = pd.read_csv("/kaggle/working/interaction.csv",encoding="latin1")
protein_df = pd.read_csv("/kaggle/working/protein.csv",encoding="latin1")
#the positive samples are validated exprimentally , but the negative ones are made by switching the positions of proteins and aptamers indexes

In [ ]:
#data overview
print("=== APTAMER ===")
print(aptamer_df.shape)
print(aptamer_df.columns.tolist())
print(aptamer_df.head(3))

print("\n=== INTERACTION ===")
print(interaction_df.shape)
print(interaction_df.columns.tolist())
print(interaction_df.head(3))

print("\n=== PROTEIN ===")
print(protein_df.shape)
print(protein_df.columns.tolist())
print(protein_df.head(3))

=== APTAMER ===
(1293, 13)
['Apta_index', 'Sequence', 'Aptamer description', 'Aptamer Chemistry', 'Length', 'GC content', 'Molecular weight', 'Molarity of 1 ¦Ìg/¦Ìl solution', 'Number of tetrads', 'G-Score', 'Function', 'Applications', 'Assay']
  Apta_index                                           Sequence  \
0     Apta_1              ACGAAGCUUGAUCCCGUUUGCCGGUCGAUCGCUUCGA   
1     Apta_2  GGGCUUCUCUGGUUAGACCAGAUUUGAGCCUGGGAGCUCUCUGGCU...   
2     Apta_3        CCTGCCTTTGGCAGCCGGGCGAAGGCAACCCGACACCGACAGC   

  Aptamer description Aptamer Chemistry Length GC content Molecular weight  \
0             RNA Tat               RNA  37 nt      0.568     11,783.98 Da   
1               TAR-1               RNA  59 nt      0.559     18,943.21 Da   
2         Aptamer A14               DNA  43 nt      0.698     13,169.44 Da   

  Molarity of 1 ¦Ìg/¦Ìl solution Number of tetrads  G-Score  \
0                      84.86 ¦ÌM     No QGRS found      NaN   
1                      52.79 ¦ÌM     No QGRS fo

In [ ]:



#there was space issues
aptamer_df.columns = aptamer_df.columns.str.strip()
interaction_df.columns = interaction_df.columns.str.strip()
protein_df.columns = protein_df.columns.str.strip()
#everything has to be merged cause we have 3 different files 
# --------- positive pairs 
positives = (
    interaction_df[interaction_df["Target chemistry"] == "Protein"][["Apta_index", "TargetID"]]
    .merge(aptamer_df[["Apta_index", "Sequence", "Aptamer Chemistry"]], on="Apta_index")
    .merge(protein_df[["Uni-port ID", "Protein names"]], left_on="TargetID", right_on="Uni-port ID")
    [["Sequence", "TargetID", "Protein names", "Aptamer Chemistry"]]
    .rename(columns={"Sequence": "aptamer_seq"})
)
positives["label"] = 1  # the 1 is mean positive pauiring

# -------- negative pairs (random shuffle) 
target_ids = list(positives["TargetID"])
negatives = positives.copy()
negatives["TargetID"] = target_ids[1:] + [target_ids[0]]  #we shufle 
negatives["label"] = 0  #negative pair 

dataset = pd.concat([positives, negatives]).sample(frac=1, random_state=42).reset_index(drop=True)
print(dataset.shape)
print(dataset["label"].value_counts())
print(dataset.head(5))




(1278, 5)
label
1    639
0    639
Name: count, dtype: int64
                                         aptamer_seq TargetID  \
0                               AGGAAGGCTTTAGGTGGAAT   P04585   
1  GGGAGACAAGAAUAAACGCUCAACGUUCAGUAUAACAGUCCGAGUC...   Q05127   
2                          TGTGGGGGTGGACGGGCCGGGTAGA   P62805   
3                         GGGTTGGAGGTGGAGGGGAGTTGAGG   P14210   
4  CCGTATCGCCCAGGCAACTGGGCTAAACTTCCCAGAGGGAACGAAA...   O46492   

                                       Protein names Aptamer Chemistry  label  
0  Gag-Pol polyprotein (Pr160Gag-Pol) [Cleaved in...               DNA      1  
1      Polymerase cofactor VP35 (Ebola VP35) (eVP35)               RNA      1  
2  Vascular endothelial growth factor A (VEGF-A) ...               DNA      0  
3  Hepatocyte growth factor (Hepatopoietin-A) (Sc...               DNA      0  
4  Aspartyl/asparaginyl beta-hydroxylase (EC 1.14...               DNA      0  


In [11]:
dataset.columns

Index(['aptamer_seq', 'TargetID', 'Protein names', 'Aptamer Chemistry',
       'label'],
      dtype='object')

In [ ]:

# we need to fetch teh actual sequence of the protein using the id of the protein
def fetch_uniprot_sequence(uniprot_id):
    url = f"https://rest.uniprot.org/uniprotkb/{uniprot_id}.fasta"
    r = requests.get(url)
    if r.status_code == 200:
        lines = r.text.strip().split("\n")
        return "".join(lines[1:]) 
    return None

protein_ids = positives["TargetID"].unique()
id_to_seq = {}

for pid in protein_ids:
    seq = fetch_uniprot_sequence(pid)
    id_to_seq[pid] = seq
    time.sleep(0.3)  # be polite to UniProt API
    print(f"{pid}: {seq[:30] if seq else 'NOT FOUND'}")

# add to dataset
positives["protein_seq"] = positives["TargetID"].map(id_to_seq)

A6MI22: MEPVDPRLEPWKHPGSQPKTACTSCYCKRC...
E0Z2R8: MKAILVVLLYTFATANADTLCIGYHANNST...
G3GC05: MAAEGSETNARNSESVRGQIFEVGPRYTNL...
I0IT65: MKTFLILALLAIVATTATSAVRVPVPQLQP...
O14786: MERGLPLLCAVLALVLAPAGAFRNDKCGDT...
O15243: MAGVKALVALSFSGAIGLTFLMLGCALEDY...
O15455: MRQTLPCIYFWGGLLPFGMLCASSTTKCTV...
O43504: MEATLEQHLEDTMKNPSIVGVLCTDSQGLN...
O60930: MSWLLFLAHRVALAALPCRRGSRGFGMFYA...
O95786: MTTEQRRSLQAFQDYIRKTLDPTYILSYMA...
O95994: MEKIPVSAFLLLVALSYTLARDTTVKPGAK...
P00432: MADNRDPASDQMKHWKEQRAAQKPDVLTTG...
P00533: MRPSGTAGAALLALLAALCPASRALEEKKV...
P00646: MSGGDGRGHNTGAHSTSGNINGGPTGLGVG...
P00698: MRSLLILVLCFLPLAALGKVFGRCELAAAM...
P00734: MAHVRGLQLPGCLALAALCSLVHSQHVFLA...
P00735: MARVRGPRLPGCLALAALFSLVHSQHVFLA...
P00740: MQRVNMIMAESPGLITICLLGYLLSAECTV...
P00749: MRALLARLLLCVLVVSDSKGSNELHQVPSN...
P01031: MGLLGILCFLIFLGKTWGQEQTYVISAPKI...
P01298: MAAARLCLSLLLLSTCVALLLQPLLGAQGA...
P01303: MLGNKRLGLSGLTLALSLLVCLGALAEAYP...
P01308: MALWMRLLPLLALLALWGPDPAAAFVNQHL...
P01375: MSTESMIRDVELAEEALPKKTGGPQG

In [ ]:

positives = positives.dropna(subset=["protein_seq"])
print(f"positives after dropping missing sequences: {len(positives)}")

# build negatives AFTER (so protein_seq column exists)
target_ids = list(positives["TargetID"])
negatives = positives.copy()
negatives["TargetID"] = target_ids[1:] + [target_ids[0]]
# remap protein_seq to match the new (shuffled) TargetID
negatives["protein_seq"] = negatives["TargetID"].map(id_to_seq)
negatives["label"] = 0

# now concat
dataset = pd.concat([positives, negatives]).sample(frac=1, random_state=42).reset_index(drop=True)

print(dataset.shape)
print(dataset["label"].value_counts())
print(dataset[["aptamer_seq", "protein_seq", "label"]].head(5))

positives after dropping missing sequences: 639
(1278, 6)
label
1    639
0    639
Name: count, dtype: int64
                                         aptamer_seq  \
0                               AGGAAGGCTTTAGGTGGAAT   
1  GGGAGACAAGAAUAAACGCUCAACGUUCAGUAUAACAGUCCGAGUC...   
2                          TGTGGGGGTGGACGGGCCGGGTAGA   
3                         GGGTTGGAGGTGGAGGGGAGTTGAGG   
4  CCGTATCGCCCAGGCAACTGGGCTAAACTTCCCAGAGGGAACGAAA...   

                                         protein_seq  label  
0  MGARASVLSGGELDRWEKIRLRPGGKKKYKLKHIVWASRELERFAV...      1  
1  MTTRTKGRGHTAATTQNDRMPGPELSGWISEQLMTGRIPVSDIFCD...      1  
2  MSGRGKGGKGLGKGGAKRHRKVLRDNIQGITKPAIRRLARRGGVKR...      0  
3  MWVTKLLPALLLQHVLLHLLLLPIAIPYAEGQRKRRNTIHEFKKSA...      0  
4  MKWLVLLGLVAFSECIVKIPLRRVKTMTKTLSGKNMLNNFVKEHAY...      0  


In [14]:
print(dataset.columns)

Index(['aptamer_seq', 'TargetID', 'Protein names', 'Aptamer Chemistry',
       'label', 'protein_seq'],
      dtype='object')


In [ ]:


# --- encoders : we need to generate the embedding cause this is what clip works with : we use the same ones i used for the generator architecture 
nt_tokenizer = AutoTokenizer.from_pretrained("InstaDeepAI/nucleotide-transformer-500m-human-ref")
nt_model = AutoModel.from_pretrained("InstaDeepAI/nucleotide-transformer-500m-human-ref")

esm_tokenizer = EsmTokenizer.from_pretrained("facebook/esm2_t6_8M_UR50D")  # small ESM2
esm_model = EsmModel.from_pretrained("facebook/esm2_t6_8M_UR50D")

config.json:   0%|          | 0.00/706 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/129 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/101 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.94G [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/775 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/31.4M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/107 [00:00<?, ?it/s]

EsmModel LOAD REPORT from: facebook/esm2_t6_8M_UR50D
Key                         | Status     | 
----------------------------+------------+-
esm.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
pooler.dense.weight         | MISSING    | 
pooler.dense.bias           | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
print(len(dataset))

train_df, test_df = train_test_split(
    dataset,
    test_size=0.2,
    random_state=42,
    shuffle=True
)
print(len(train_df),len(test_df))

1278
1022 256


In [ ]:
for p in nt_model.parameters(): p.requires_grad = False
for p in esm_model.parameters(): p.requires_grad = False
# i am not fine tuning the encoders of the embeddings

In [ ]:


# ------- classifier head  
class AptaBindingClassifier(nn.Module):
    """
    Clip model to work as evaluator for the generated aptamer
    Input : the proteni embedding   +  the aptamer (generated)  embedding
    Output : 0/1  : bind/no bind 
    """
    def __init__(self, apt_dim=1280, prot_dim=320, hidden=512):
        super().__init__()
        self.classifier = nn.Sequential(
            nn.Linear(apt_dim + prot_dim, hidden),  # 1280+320=1600 ---> 512    
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(hidden, 1),  #512 --->1   the probablity of binding 
            nn.Sigmoid()   # to convert the prob to bindary 
        )
    
    def forward(self, apt_emb, prot_emb):
        x = torch.cat([apt_emb, prot_emb], dim=-1)
        return self.classifier(x).squeeze(-1)

def get_embedding(model, tokenizer, seqs, max_len=512, device="cpu"):
    """
    Geenrate the embedding of the protein (original input) and the generated aptamer
    """
    tokens = tokenizer(seqs, return_tensors="pt", padding=True,
                       truncation=True, max_length=max_len).to(device)
    with torch.no_grad():
        out = model(**tokens)
    # mean pool instead of CLS
    mask = tokens["attention_mask"].unsqueeze(-1).float()
    return (out.last_hidden_state * mask).sum(1) / mask.sum(1)


class AptaDataset(Dataset):
    def __init__(self, df):
        self.apt = list(df["aptamer_seq"])
        self.prot = list(df["protein_seq"])
        self.labels = torch.tensor(df["label"].values, dtype=torch.float32)
    def __len__(self): return len(self.labels)
    def __getitem__(self, i): return self.apt[i], self.prot[i], self.labels[i]

device = "cuda" if torch.cuda.is_available() else "cpu"
nt_model.to(device); esm_model.to(device)

model = AptaBindingClassifier(apt_dim=1280, prot_dim=320).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
criterion = nn.BCELoss()

loader = DataLoader(AptaDataset(train_df), batch_size=8, shuffle=True)



In [ ]:
# test 
sample_apt = [dataset["aptamer_seq"].iloc[0]]
sample_prot = [dataset["protein_seq"].iloc[0]]

apt_emb = get_embedding(nt_model, nt_tokenizer, sample_apt, device=device)
prot_emb = get_embedding(esm_model, esm_tokenizer, sample_prot, device=device)

print("Aptamer embedding dim:", apt_emb.shape)   # nt hidden size
print("Protein embedding dim:", prot_emb.shape)  # esm2 hidden size
print("Total:", apt_emb.shape[-1] + prot_emb.shape[-1])

Aptamer embedding dim: torch.Size([1, 1280])
Protein embedding dim: torch.Size([1, 320])
Total: 1600


In [ ]:
#training loop
for epoch in range(50):
    model.train()
    total_loss = 0
    for apt_seqs, prot_seqs, labels in loader:
        labels = labels.to(device)
        apt_emb = get_embedding(nt_model, nt_tokenizer, list(apt_seqs), device=device)
        prot_emb = get_embedding(esm_model, esm_tokenizer, list(prot_seqs), device=device)
        
        preds = model(apt_emb, prot_emb)
        loss = criterion(preds, labels)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    
    print(f"Epoch {epoch+1} | Loss: {total_loss/len(loader):.4f}")

Epoch 1 | Loss: 0.6946
Epoch 2 | Loss: 0.6952
Epoch 3 | Loss: 0.6945
Epoch 4 | Loss: 0.6908
Epoch 5 | Loss: 0.6921
Epoch 6 | Loss: 0.6937
Epoch 7 | Loss: 0.6907
Epoch 8 | Loss: 0.6962
Epoch 9 | Loss: 0.6909
Epoch 10 | Loss: 0.6877
Epoch 11 | Loss: 0.6891
Epoch 12 | Loss: 0.6895
Epoch 13 | Loss: 0.6919
Epoch 14 | Loss: 0.6863
Epoch 15 | Loss: 0.6885
Epoch 16 | Loss: 0.6868
Epoch 17 | Loss: 0.6905
Epoch 18 | Loss: 0.6901
Epoch 19 | Loss: 0.6858
Epoch 20 | Loss: 0.6841
Epoch 21 | Loss: 0.6897
Epoch 22 | Loss: 0.6870
Epoch 23 | Loss: 0.6850
Epoch 24 | Loss: 0.6860
Epoch 25 | Loss: 0.6810
Epoch 26 | Loss: 0.6848
Epoch 27 | Loss: 0.6857
Epoch 28 | Loss: 0.6890
Epoch 29 | Loss: 0.6807
Epoch 30 | Loss: 0.6847
Epoch 31 | Loss: 0.6788
Epoch 32 | Loss: 0.6799
Epoch 33 | Loss: 0.6834
Epoch 34 | Loss: 0.6830
Epoch 35 | Loss: 0.6806
Epoch 36 | Loss: 0.6763
Epoch 37 | Loss: 0.6809
Epoch 38 | Loss: 0.6740
Epoch 39 | Loss: 0.6769
Epoch 40 | Loss: 0.6742
Epoch 41 | Loss: 0.6816
Epoch 42 | Loss: 0.6757
E

In [ ]:

model.eval()

all_preds = []
all_probs = []
all_labels = []

test_loader = DataLoader(AptaDataset(test_df), batch_size=8, shuffle=False)

with torch.no_grad():
    for apt_seqs, prot_seqs, labels in test_loader:
        labels = labels.to(device)
        
        apt_emb = get_embedding(nt_model, nt_tokenizer, list(apt_seqs), device=device)
        prot_emb = get_embedding(esm_model, esm_tokenizer, list(prot_seqs), device=device)
        
        probs = model(apt_emb, prot_emb)
        preds = (probs >= 0.5).float()
        
        all_probs.extend(probs.cpu().numpy())
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

print("=== Test Results ===")
print(f"Accuracy:  {accuracy_score(all_labels, all_preds):.4f}")
print(f"ROC-AUC:   {roc_auc_score(all_labels, all_probs):.4f}")
print()
print("Classification Report:")
print(classification_report(all_labels, all_preds, target_names=["Non-binding", "Binding"]))
print()
print("Confusion Matrix:")
print(confusion_matrix(all_labels, all_preds))

=== Test Results ===
Accuracy:  0.2383
ROC-AUC:   0.1057

Classification Report:
              precision    recall  f1-score   support

 Non-binding       0.32      0.42      0.36       132
     Binding       0.07      0.05      0.06       124

    accuracy                           0.24       256
   macro avg       0.20      0.23      0.21       256
weighted avg       0.20      0.24      0.21       256


Confusion Matrix:
[[ 55  77]
 [118   6]]
